# nnUNet Pipeline — Local GPU Runner

Dataset: **Dataset777_GCEF** | Samples: **mishmar_hanegev_Cu011_samp_2_Rec_nlm**, **nlm_volume** | Trainer: **nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss** | Config: **3d_fullres**

Prerequisites (run locally before this notebook):
1. `preprocessing_nnUNet_train.py` → produces `Dataset777_GCEF/` in `nnUNet_raw`
2. (For inference) `preprocessing_nnUNet_predict_tif.py` + `preprocessing_nnUNet_predict_split.py` → produces split chunks

Paths are resolved from `analysis/data_registry.json`. Do **not** edit Section 3 paths manually — change the registry instead.

Training samples loaded from registry: all samples with `status == "annotations_ready"`.


## 1) Runtime Setup

In [1]:
# Verify GPU runtime
!nvidia-smi

Sun May 24 19:25:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.61                 Driver Version: 572.61         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000             WDDM  |   00000000:3B:00.0 Off |                    0 |
| 30%   33C    P8             15W /  300W |    3359MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import os

# Local repository directory
REPO_DIR = r'C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT'
assert os.path.isdir(REPO_DIR), f'Repo not found: {REPO_DIR}'
print('Repo dir:', os.listdir(REPO_DIR))


Repo dir: ['.git', '.github', '.gitignore', '.vscode', 'analysis', 'chunk_extractor.py', 'colab_nnUNet_pipeline.ipynb', 'dataset_info.json', 'debug_labels_777.py', 'debug_labels_777_followup.py', 'extract_trainlog.py', 'Figures', 'Fiji_macros', 'inspect_predictions.py', 'legacy', 'LICENSE', 'litreture', 'make_annotations.py', 'make_psd_plot.py', 'merge_annotations.py', 'mishmar_psd.log', 'nnUNetTrainer_betterIgnoreSampling.py', 'otsu_threshold_3d.py', 'postprocessing_nnUNet_predict.py', 'postprocessing_nnUNet_predict_concatenate.py', 'postprocessing_pipeline.ipynb', 'preprocess', 'preprocessing_nnUNet_predict.py', 'preprocessing_nnUNet_predict_split.py', 'preprocessing_nnUNet_predict_tif.py', 'preprocessing_nnUNet_train.py', 'preprocess_playground', 'README.md', 'retrieve_dice_score.py', 'run_remaining_fullctx_overnight.ipynb', 'select_slices_and_predict.py', 'setup_prompt.md', 'training_diag', 'Utilities', '__path__.py', '__pycache__']


## 2) Register Custom Trainer

In [3]:
import shutil
import sys
import importlib
import nnunetv2
import os

# Find nnunetv2 trainers directory
nnunet_trainers_dir = os.path.join(
    os.path.dirname(nnunetv2.__file__),
    'training', 'nnUNetTrainer', 'variants', 'sampling'
)
os.makedirs(nnunet_trainers_dir, exist_ok=True)

src = os.path.join(REPO_DIR, 'nnUNetTrainer_betterIgnoreSampling.py')
dst = os.path.join(nnunet_trainers_dir, 'nnUNetTrainer_betterIgnoreSampling.py')
shutil.copy2(src, dst)
print(f'Copied: {src} → {dst}')

# Force reload the module to pick up the updated file
if 'nnunetv2.training.nnUNetTrainer.variants.sampling.nnUNetTrainer_betterIgnoreSampling' in sys.modules:
    del sys.modules['nnunetv2.training.nnUNetTrainer.variants.sampling.nnUNetTrainer_betterIgnoreSampling']

# Import and verify the new trainer
from nnunetv2.training.nnUNetTrainer.variants.sampling.nnUNetTrainer_betterIgnoreSampling import (
    nnUNetTrainer_betterIgnoreSampling,
    nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss
)
print('✓ Custom trainer registered:', nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss.__name__)
print('✓ Base trainer registered:', nnUNetTrainer_betterIgnoreSampling.__name__)

Copied: C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\nnUNetTrainer_betterIgnoreSampling.py → c:\Users\rony.schwartz\.conda\envs\venv-napari\Lib\site-packages\nnunetv2\training\nnUNetTrainer\variants\sampling\nnUNetTrainer_betterIgnoreSampling.py
✓ Custom trainer registered: nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss
✓ Base trainer registered: nnUNetTrainer_betterIgnoreSampling


## 3) Set Environment Variables & Paths

In [ ]:
import os, json

# ── Data Registry ──────────────────────────────────────────────────────────────
REGISTRY_PATH = os.path.join(REPO_DIR, 'analysis', 'data_registry.json')
with open(REGISTRY_PATH) as _f:
    _registry = json.load(_f)

# All samples with annotations ready — these will be staged and trained together
TRAINING_SAMPLES = [
    s for s in _registry['samples']
    if s.get('status') == 'annotations_ready'
]
assert TRAINING_SAMPLES, 'No samples with status=annotations_ready found in registry'
print(f'Training samples ({len(TRAINING_SAMPLES)}):')
for s in TRAINING_SAMPLES:
    print(f"  {s['sample_id']}")
    print(f"    raw_tiff:   {s['raw_tiff_path']}")
    print(f"    annotation: {_registry['annotations']['active_latest'].get(s['sample_id'], 'MISSING')}")

# ── Primary sample for single-volume ops (inference, etc.) ────────────────────
# Override SAMPLE_ID here if you want inference on a specific sample.
SAMPLE_ID = TRAINING_SAMPLES[0]['sample_id']
_sample_rec  = TRAINING_SAMPLES[0]

# Canonical paths from registry (all on HIVE)
RAW_TIFF_PATH    = _sample_rec['raw_tiff_path']
ANNOTATION_PATH  = _registry['annotations']['active_latest'][SAMPLE_ID]
RAW_TIFF_DIR     = os.path.dirname(RAW_TIFF_PATH)
ANNOTATION_DIR   = os.path.dirname(ANNOTATION_PATH)

print(f'\nPrimary sample (for inference): {SAMPLE_ID}')

# ── nnUNet workspace (HIVE) — shared across samples ───────────────────────────
HIVE_BASE   = r'\\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources'
LOCAL_BASE  = os.path.join(HIVE_BASE, 'multi_sample_iter01')

TRAINER_NAME = 'nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss'

nnUNet_raw          = os.path.join(LOCAL_BASE, 'nnUNet_raw')
nnUNet_preprocessed = os.path.join(LOCAL_BASE, 'nnUNet_preprocessed')
nnUNet_results      = os.path.join(LOCAL_BASE, 'nnUNet_results')

os.environ['nnUNet_raw']          = nnUNet_raw
os.environ['nnUNet_preprocessed'] = nnUNet_preprocessed
os.environ['nnUNet_results']      = nnUNet_results
os.environ['nnUNet_compile']      = 'false'   # Triton not available on Windows

for d in [nnUNet_raw, nnUNet_preprocessed, nnUNet_results]:
    os.makedirs(d, exist_ok=True)

print()
print('LOCAL_BASE:',          LOCAL_BASE)
print('nnUNet_raw:',          nnUNet_raw)
print('nnUNet_preprocessed:', nnUNet_preprocessed)
print('nnUNet_results:',      nnUNet_results)
print('nnUNet_compile:',      os.environ['nnUNet_compile'])
print('TRAINER_NAME:',        TRAINER_NAME)


Sample:          mishmar_hanegev_Cu011_samp_2_Rec_nlm
Raw TIF:         \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
Raw TIF dir:     \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5
Annotation:      \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\annotations\mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
Annotation dir:  \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\annotations

LOCAL_BASE: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm
nnUNet_raw: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\nnUNet_raw
nnUNet_preprocessed: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\nnUNet_preprocessed
nnUNet_results: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\nnUNet_results
nnUNet_compi

## 4) Verify Training Data

This notebook stages and preprocesses all `annotations_ready` samples loaded from the registry.

Expected output under HIVE (`LOCAL_BASE = multi_sample_iter01`):
```
<LOCAL_BASE>\nnUNet_raw\Dataset777_GCEF\
  imagesTr\mishmar_hanegev_Cu011_samp_2_Rec_nlm_0000.nii.gz
  imagesTr\nlm_volume_0000.nii.gz
  labelsTr\mishmar_hanegev_Cu011_samp_2_Rec_nlm.nii.gz
  labelsTr\nlm_volume.nii.gz
  dataset.json
```


In [ ]:
import subprocess, sys, os, glob, shutil

# Stage ALL training samples into a shared staging directory
stage_root   = os.path.join(LOCAL_BASE, '_multi_sample_stage')
stage_images = os.path.join(stage_root, 'images')
stage_masks  = os.path.join(stage_root, 'annotations')
os.makedirs(stage_images, exist_ok=True)
os.makedirs(stage_masks,  exist_ok=True)

# Build sets of expected filenames so we can detect stale files from removed samples
expected_image_names = set()
expected_mask_names  = set()

for s in TRAINING_SAMPLES:
    sid       = s['sample_id']
    raw_tif   = s['raw_tiff_path']
    ann_tif   = _registry['annotations']['active_latest'][sid]

    expected_image_names.add(os.path.basename(raw_tif))
    expected_mask_names.add(os.path.basename(ann_tif))

    dst_img = os.path.join(stage_images, os.path.basename(raw_tif))
    dst_msk = os.path.join(stage_masks,  os.path.basename(ann_tif))

    shutil.copy2(raw_tif, dst_img)
    shutil.copy2(ann_tif, dst_msk)
    print(f'Staged [{sid}]')
    print(f'  image: {dst_img}')
    print(f'  mask:  {dst_msk}')

# Remove any stale files left over from previous runs with different samples
for p in glob.glob(os.path.join(stage_images, '*.tif')) + glob.glob(os.path.join(stage_images, '*.tiff')):
    if os.path.basename(p) not in expected_image_names:
        os.remove(p)
        print(f'Removed stale image: {p}')
for p in glob.glob(os.path.join(stage_masks, '*.tif')) + glob.glob(os.path.join(stage_masks, '*.tiff')):
    if os.path.basename(p) not in expected_mask_names:
        os.remove(p)
        print(f'Removed stale mask: {p}')

# Patch __path__.py to point to the multi-sample staging dirs
_path_py = os.path.join(REPO_DIR, '__path__.py')
_path_py_content = (
    f'PATH_ImageJ = ""\n'
    f'PATH_nnUNet_raw = r"{nnUNet_raw}"\n'
    f'input_dir_images = r"{stage_images}"\n'
    f'input_dir_masks = r"{stage_masks}"\n'
)
with open(_path_py, 'w') as _f:
    _f.write(_path_py_content)
print('\nPatched __path__.py:')
print(_path_py_content)

# Determine whether a rebuild is needed
expected_imgs = sorted(f'{s["sample_id"]}_0000.nii.gz' for s in TRAINING_SAMPLES)
expected_lbls = sorted(f'{s["sample_id"]}.nii.gz'       for s in TRAINING_SAMPLES)
dataset_raw_dir  = os.path.join(nnUNet_raw, 'Dataset777_GCEF')
images_tr_dir    = os.path.join(dataset_raw_dir, 'imagesTr')
labels_tr_dir    = os.path.join(dataset_raw_dir, 'labelsTr')

needs_rebuild = True
if os.path.isdir(images_tr_dir) and os.path.isdir(labels_tr_dir):
    existing_imgs = sorted(os.path.basename(p) for p in glob.glob(os.path.join(images_tr_dir, '*_0000.nii.gz')))
    existing_lbls = sorted(os.path.basename(p) for p in glob.glob(os.path.join(labels_tr_dir, '*.nii.gz')))
    if existing_imgs == expected_imgs and existing_lbls == expected_lbls:
        needs_rebuild = False
        print('Dataset already prepared for all training samples; skipping preprocessing.')
    else:
        print(f'Dataset mismatch — expected {expected_imgs}, found {existing_imgs}. Rebuilding.')
        shutil.rmtree(dataset_raw_dir)

if needs_rebuild:
    print('Running preprocessing_nnUNet_train.py for multi-sample staged dirs...')
    result = subprocess.run(
        [sys.executable, os.path.join(REPO_DIR, 'preprocessing_nnUNet_train.py')],
        cwd=REPO_DIR,
        env=os.environ,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:\n', result.stderr)
        raise RuntimeError(f'preprocessing_nnUNet_train.py failed (exit {result.returncode})')
    print('Preprocessing done.')


Staged image: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\_single_sample_stage\images\mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
Staged mask:  \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\_single_sample_stage\annotations\mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
Patched __path__.py:
PATH_ImageJ = ""
PATH_nnUNet_raw = r"\\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\nnUNet_raw"
input_dir_images = r"\\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\_single_sample_stage\images"
input_dir_masks = r"\\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\_single_sample_stage\annotations"

Dataset already prepared as single-sample; skipping preprocessing.


In [ ]:
# Verify all training samples are present in the dataset
dataset_dir = os.path.join(nnUNet_raw, 'Dataset777_GCEF')
assert os.path.isdir(dataset_dir), f'Dataset folder not found: {dataset_dir}'

import glob, os
images = sorted(glob.glob(os.path.join(dataset_dir, 'imagesTr', '*_0000.nii.gz')))
labels = sorted(glob.glob(os.path.join(dataset_dir, 'labelsTr', '*.nii.gz')))
dataset_json = os.path.join(dataset_dir, 'dataset.json')

print(f'imagesTr: {len(images)} files')
print(f'labelsTr: {len(labels)} files')
print(f'dataset.json exists: {os.path.isfile(dataset_json)}')
for p in images:
    print('  image:', os.path.basename(p))
for p in labels:
    print('  label:', os.path.basename(p))

expected_imgs = sorted(f'{s["sample_id"]}_0000.nii.gz' for s in TRAINING_SAMPLES)
expected_lbls = sorted(f'{s["sample_id"]}.nii.gz'       for s in TRAINING_SAMPLES)

assert [os.path.basename(p) for p in images] == expected_imgs, f'imagesTr mismatch: {[os.path.basename(p) for p in images]} != {expected_imgs}'
assert [os.path.basename(p) for p in labels] == expected_lbls, f'labelsTr mismatch: {[os.path.basename(p) for p in labels]} != {expected_lbls}'
assert os.path.isfile(dataset_json), 'dataset.json missing'
print(f'✓ Dataset contains all {len(TRAINING_SAMPLES)} training samples.')


imagesTr: 1 files
labelsTr: 1 files
dataset.json exists: True
  image: mishmar_hanegev_Cu011_samp_2_Rec_nlm_0000.nii.gz
  label: mishmar_hanegev_Cu011_samp_2_Rec_nlm.nii.gz
✓ Dataset is restricted to target sample only.


## 5) nnUNet Planning & Preprocessing

In [15]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, '-m', 'nnunetv2.experiment_planning.plan_and_preprocess_entrypoints',
     '-d', '777', '--verify_dataset_integrity'],
    env=os.environ,
    timeout=3600,  # 1 hour timeout
    capture_output=False  # Don't stream, just let it run
)

if result.returncode != 0:
    raise RuntimeError(f"plan_and_preprocess failed (exit {result.returncode})")
print("✓ Planning and preprocessing complete.")


✓ Planning and preprocessing complete.


## 6) Training

Run one fold at a time. Change `FOLD` to train multiple folds sequentially (0–4).

**Warning**: each fold may take many hours on a single Colab GPU.

In [8]:
# PolyLRScheduler compatibility fix is applied directly to polylr.py on disk.
# No runtime patch needed here.
print("PolyLRScheduler: using patched polylr.py (PyTorch 2.x compatible)")


PolyLRScheduler: using patched polylr.py (PyTorch 2.x compatible)


In [ ]:
# Create custom splits_final.json — all training samples in both train and val
import json, os

preprocessed_dir = os.path.join(os.environ['nnUNet_preprocessed'], 'Dataset777_GCEF')
sample_ids = [s['sample_id'] for s in TRAINING_SAMPLES]
splits = [{"train": sample_ids, "val": sample_ids}]

splits_path = os.path.join(preprocessed_dir, 'splits_final.json')
with open(splits_path, 'w') as f:
    json.dump(splits, f, indent=2)

print(f"Wrote {splits_path}")
print(json.dumps(splits, indent=2))


Wrote \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\nnUNet_preprocessed\Dataset777_GCEF\splits_final.json
[
  {
    "train": [
      "mishmar_hanegev_Cu011_samp_2_Rec_nlm"
    ],
    "val": [
      "mishmar_hanegev_Cu011_samp_2_Rec_nlm"
    ]
  }
]


In [17]:
import subprocess, sys, os

# Early stopping controls (env-driven in trainer)
os.environ['NNUNET_EARLY_STOP_ENABLED'] = '1'
os.environ['NNUNET_EARLY_STOP_PATIENCE'] = '20'
os.environ['NNUNET_EARLY_STOP_MIN_DELTA'] = '0.001'
print('Early stopping env:', {
    'NNUNET_EARLY_STOP_ENABLED': os.environ['NNUNET_EARLY_STOP_ENABLED'],
    'NNUNET_EARLY_STOP_PATIENCE': os.environ['NNUNET_EARLY_STOP_PATIENCE'],
    'NNUNET_EARLY_STOP_MIN_DELTA': os.environ['NNUNET_EARLY_STOP_MIN_DELTA'],
})

# Stream training output live to the notebook cell
proc = subprocess.Popen(
    [sys.executable, '-m', 'nnunetv2.run.run_training',
     '777', '3d_fullres', '0', '-tr', TRAINER_NAME],
    env=os.environ,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # merge stderr into stdout
    text=True,
    bufsize=1
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"run_training failed (exit {proc.returncode})")


Early stopping env: {'NNUNET_EARLY_STOP_ENABLED': '1', 'NNUNET_EARLY_STOP_PATIENCE': '20', 'NNUNET_EARLY_STOP_MIN_DELTA': '0.001'}

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0
c:\Users\rony.schwartz\.conda\envs\venv-napari\Lib\site-packages\nnunetv2\training\nnUNetTrainer\nnUNetTrainer.py:161: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.grad_scaler = GradScaler() if self.device.type == 'cuda' else None

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for d

In [18]:
# List training results
results_dir = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)
if os.path.isdir(results_dir):
    for item in sorted(os.listdir(results_dir)):
        print(item)
else:
    print(f'Results dir not found yet: {results_dir}')

dataset.json
dataset_fingerprint.json
fold_0
plans.json


## 7) Training Outputs

Results are saved on HIVE at:
```
<LOCAL_BASE>\nnUNet_results\Dataset777_GCEF\<TRAINER_NAME>__nnUNetPlans__3d_fullres\
```
Use `extract_trainlog.py` to parse training logs and plot loss curves.


In [19]:
# Optional: list training result files
results_dir = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)
if os.path.isdir(results_dir):
    for item in sorted(os.listdir(results_dir)):
        print(item)
else:
    print(f'Results dir not found yet: {results_dir}')


dataset.json
dataset_fingerprint.json
fold_0
plans.json


## 8) Inference Data

Inference is run per-sample. Set `INFERENCE_SAMPLE_ID` below to select which volume to predict.
Available samples: all entries in `TRAINING_SAMPLES`.

Split chunks are written to:
`<HIVE_BASE>\<INFERENCE_SAMPLE_ID>\inference_input\`


In [ ]:
import subprocess, sys, os, glob
import numpy as np
import tifffile
import nibabel as nib
from tqdm import tqdm

# ── Select inference sample ────────────────────────────────────────────────────
# Change INFERENCE_SAMPLE_ID to run inference on a different sample.
INFERENCE_SAMPLE_ID = TRAINING_SAMPLES[0]['sample_id']
_inf_rec = next(s for s in TRAINING_SAMPLES if s['sample_id'] == INFERENCE_SAMPLE_ID)
INF_RAW_TIFF_PATH = _inf_rec['raw_tiff_path']

# Per-sample inference workspace (under HIVE, NOT under multi_sample LOCAL_BASE)
INF_SAMPLE_BASE = os.path.join(HIVE_BASE, INFERENCE_SAMPLE_ID)
NIFTI_DIR       = os.path.join(INF_SAMPLE_BASE, 'nifti_predict')
INFERENCE_INPUT = os.path.join(INF_SAMPLE_BASE, 'inference_input')
os.makedirs(NIFTI_DIR,       exist_ok=True)
os.makedirs(INFERENCE_INPUT, exist_ok=True)

# Model folder for plans.json (from the multi-sample training workspace)
MODEL_DIR = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)

print(f'Inference sample:  {INFERENCE_SAMPLE_ID}')
print(f'Raw TIF:           {INF_RAW_TIFF_PATH}')
print(f'NIfTI dir:         {NIFTI_DIR}')
print(f'Inference input:   {INFERENCE_INPUT}')
print(f'Model dir:         {MODEL_DIR}')

# --- Step 1: .tif -> _0000.nii.gz (direct Python, zscore norm) ---
print('\n=== Step 1: .tif -> _0000.nii.gz ===')
vol = tifffile.imread(INF_RAW_TIFF_PATH).astype(np.float32)
mean, std = vol.mean(), vol.std()
vol = (vol - mean) / (std + 1e-8)
vol = vol.transpose(2, 1, 0)  # (Z, Y, X) -> (X, Y, Z) for nibabel
stem = os.path.splitext(os.path.basename(INF_RAW_TIFF_PATH))[0]
out_path = os.path.join(NIFTI_DIR, f'{stem}_0000.nii.gz')
nib.save(nib.Nifti1Image(vol, affine=np.eye(4)), out_path)
print(f'  Saved: {out_path}  shape={vol.shape}')
print('Step 1 done.')

# --- Step 2: _0000.nii.gz -> split chunks -> INFERENCE_INPUT ---
print('=== Step 2: preprocessing_nnUNet_predict_split.py ===')
result = subprocess.run(
    [sys.executable,
     os.path.join(REPO_DIR, 'preprocessing_nnUNet_predict_split.py'),
     '-i', NIFTI_DIR,
     '-o', INFERENCE_INPUT,
     '-m', MODEL_DIR],
    env=os.environ, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:\n', result.stderr)
    raise RuntimeError(f'preprocessing_nnUNet_predict_split.py failed (exit {result.returncode})')
print('Step 2 done.')


TIF_INPUT (from registry): \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5
=== Step 1: .tif -> _0000.nii.gz (direct Python, zscore norm) ===


Converting TIF:   0%|          | 0/1 [00:00<?, ?it/s]

Converting TIF: 100%|██████████| 1/1 [01:45<00:00, 105.14s/it]

  Saved: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\nifti_predict\mishmar_hanegev_Cu011_samp_2_Rec_nlm_0000.nii.gz  shape=(650, 650, 822)
Step 1 done.
=== Step 2: preprocessing_nnUNet_predict_split.py ===


Model Parameters:
Patch Size:     [128, 160, 112]
Target Spacing: [1.0, 1.0, 1.0]
----------
1 Images found
----------
Image File:    mishmar_hanegev_Cu011_samp_2_Rec_nlm_0000.nii.gz
Image Shape:   (650, 650, 822)
Image Spacing: (1.0, 1.0, 1.0)
Original Patch Size:   [128. 160. 112.]
Patch Overlap:   [64. 80. 56.]
Base Crop Size:   [650 650 103]
 - Split 0 from [0 0 0] to [650 650 159]
 - Split 1 from [ 0  0 47] to [650 650 262]
 - Split 2 from [  0   0 150] to [650 650 365]
 - Split 3 from [  0   0 253] to [650 650 468]
 - Split 4 from [  0   0 356] to [650 650 571]
 - Split 5 from [  0   0 459] to [650 650 674]
 - Split 6 from [  0   0 562] to [650 650 777]
 - Split 7 from [  0   0 665] to [650 650 822]

Step 2 done.


In [ ]:
INFERENCE_OUTPUT = os.path.join(INF_SAMPLE_BASE, 'inference_output')
os.makedirs(INFERENCE_OUTPUT, exist_ok=True)

assert os.path.isdir(INFERENCE_INPUT), f'Inference input not found: {INFERENCE_INPUT}'
input_files = [f for f in os.listdir(INFERENCE_INPUT) if f.endswith('_0000.nii.gz')]
print(f'Sample:                  {INFERENCE_SAMPLE_ID}')
print(f'Inference input dir:     {INFERENCE_INPUT}')
print(f'Inference output dir:    {INFERENCE_OUTPUT}')
print(f'Inference chunks found:  {len(input_files)}')


Inference chunks found: 8


## 9) Inference

In [22]:
import os

# Diagnose: what datasets exist in nnUNet_results?
print('=== nnUNet_results contents ===')
if os.path.isdir(nnUNet_results):
    datasets = os.listdir(nnUNet_results)
    if datasets:
        for d in sorted(datasets):
            print(f'  {d}')
            sub = os.path.join(nnUNet_results, d)
            for item in sorted(os.listdir(sub)):
                print(f'    {item}')
    else:
        print('  (empty — no training results found)')
else:
    print(f'  nnUNet_results dir does not exist: {nnUNet_results}')

# Check specifically for the expected fold checkpoint
expected = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres',
    'fold_0', 'checkpoint_final.pth'
)
print(f'\nExpected checkpoint exists: {os.path.isfile(expected)}')
print(f'Expected checkpoint path:   {expected}')


=== nnUNet_results contents ===
  Dataset777_GCEF
    nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres

Expected checkpoint exists: True
Expected checkpoint path:   \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres\fold_0\checkpoint_final.pth


In [23]:
import os
import torch
import numpy
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

# nnUNet checkpoints were saved with PyTorch <2.6, which used weights_only=False.
# PyTorch 2.6 changed the default to True, breaking checkpoint loading.
# Patch torch.load to restore weights_only=False (safe: this is our own trained model).
_orig_torch_load = torch.load
def _patched_load(f, *args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_torch_load(f, *args, **kwargs)
torch.load = _patched_load

MODEL_DIR = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)
print(f'Model dir: {MODEL_DIR}')
print(f'CUDA available: {torch.cuda.is_available()}')

# Auto-select checkpoint: prefer final, fall back to best, then latest
fold_dir = os.path.join(MODEL_DIR, 'fold_0')
for candidate in ('checkpoint_final.pth', 'checkpoint_best.pth', 'checkpoint_latest.pth'):
    if os.path.isfile(os.path.join(fold_dir, candidate)):
        checkpoint_name = candidate
        break
else:
    raise FileNotFoundError(f'No checkpoint found in {fold_dir}')
print(f'Using checkpoint: {checkpoint_name}')

# More stable inference settings for notebooks/Windows to avoid kernel crashes.
# Keep CUDA for compute when available, but do preprocessing/postprocessing off-device.
use_cuda = torch.cuda.is_available()
device = torch.device('cuda' if use_cuda else 'cpu')
perform_everything_on_device = False

if use_cuda:
    torch.cuda.empty_cache()

predictor = nnUNetPredictor(
    tile_step_size=0.5,
    use_gaussian=True,
    use_mirroring=False,
    perform_everything_on_device=perform_everything_on_device,
    device=device,
    verbose=True,
    allow_tqdm=True
)

predictor.initialize_from_trained_model_folder(
    MODEL_DIR,
    use_folds=(0,),
    checkpoint_name=checkpoint_name
)

predictor.predict_from_files(
    INFERENCE_INPUT,
    INFERENCE_OUTPUT,
    save_probabilities=False,
    overwrite=True,
    num_processes_preprocessing=1,
    num_processes_segmentation_export=1,
    folder_with_segs_from_prev_stage=None,
    num_parts=1,
    part_id=0
)
print('Inference complete.')


Model dir: \\hive3065\Yael_Mishael\Rony\remote_computer backup\nnUNet_resources\mishmar_hanegev_Cu011_samp_2_Rec_nlm\nnUNet_results\Dataset777_GCEF\nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss__nnUNetPlans__3d_fullres
CUDA available: True
Using checkpoint: checkpoint_final.pth
There are 8 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 8 cases that I would like to predict

Predicting mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__0__159_:
perform_everything_on_device: False
Input shape: torch.Size([1, 159, 650, 650])
step_size: 0.5
mirror_axes: None
n_steps 160, image size is torch.Size([159, 650, 650]), tile_size [112, 160, 128], tile_step_size 0.5
steps:
[[0, 47], [0, 70, 140, 210, 280, 350, 420, 490], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522]]
move image to device cpu
preallocating results arrays on device cpu


100%|██████████| 160/160 [00:16<00:00,  9.98it/s]


Prediction done
sending off prediction to background worker for resampling and export
done with mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__0__159_

Predicting mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__150__365_:
perform_everything_on_device: False
Input shape: torch.Size([1, 215, 650, 650])
step_size: 0.5
mirror_axes: None
n_steps 240, image size is torch.Size([215, 650, 650]), tile_size [112, 160, 128], tile_step_size 0.5
steps:
[[0, 52, 103], [0, 70, 140, 210, 280, 350, 420, 490], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522]]
move image to device cpu
preallocating results arrays on device cpu


100%|██████████| 240/240 [00:22<00:00, 10.60it/s]


Prediction done
sending off prediction to background worker for resampling and export
done with mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__150__365_

Predicting mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__253__468_:
perform_everything_on_device: False
Input shape: torch.Size([1, 215, 650, 650])
step_size: 0.5
mirror_axes: None
n_steps 240, image size is torch.Size([215, 650, 650]), tile_size [112, 160, 128], tile_step_size 0.5
steps:
[[0, 52, 103], [0, 70, 140, 210, 280, 350, 420, 490], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522]]
move image to device cpu
preallocating results arrays on device cpu


...

Prediction done
sending off prediction to background worker for resampling and export
done with mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__47__262_

Predicting mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__562__777_:
perform_everything_on_device: False
Input shape: torch.Size([1, 215, 650, 650])
step_size: 0.5
mirror_axes: None
n_steps 240, image size is torch.Size([215, 650, 650]), tile_size [112, 160, 128], tile_step_size 0.5
steps:
[[0, 52, 103], [0, 70, 140, 210, 280, 350, 420, 490], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522]]
move image to device cpu
preallocating results arrays on device cpu


100%|██████████| 240/240 [00:24<00:00,  9.92it/s]


Prediction done
sending off prediction to background worker for resampling and export
done with mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__562__777_

Predicting mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__665__822_:
perform_everything_on_device: False
Input shape: torch.Size([1, 157, 650, 650])
step_size: 0.5
mirror_axes: None
n_steps 160, image size is torch.Size([157, 650, 650]), tile_size [112, 160, 128], tile_step_size 0.5
steps:
[[0, 45], [0, 70, 140, 210, 280, 350, 420, 490], [0, 58, 116, 174, 232, 290, 348, 406, 464, 522]]
move image to device cpu
preallocating results arrays on device cpu


100%|██████████| 160/160 [00:12<00:00, 13.24it/s]


Prediction done
sending off prediction to background worker for resampling and export
done with mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__665__822_
Inference complete.


## 10) Validate Predictions


In [24]:
# List prediction outputs
pred_files = [f for f in os.listdir(INFERENCE_OUTPUT) if f.endswith('.nii.gz')]
print(f'Predictions: {len(pred_files)} files')
for f in sorted(pred_files):
    print(f'  {f}')

Predictions: 8 files
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__0__159_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__150__365_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__253__468_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__356__571_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__459__674_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__47__262_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__562__777_.nii.gz
  mishmar_hanegev_Cu011_samp_2_Rec_nlm__2__665__822_.nii.gz
